<div style="text-align:center; font-family:Tahoma, Arial; line-height:1.8;">

  <div style="font-size:42px; font-weight:bold; color:#0F8298;">
    machine learning 15
  </div>

  <div style="font-size:28px; font-weight:600; color:#6C3BAA; margin-top:8px;">
    Recommender System </div>
  

  <div style="font-size:18px; color:#4b4f9c;">
</div>
content Based Filtering

پالایش مبتنی بر محتوا

<hr style="width:60%; margin:20px auto; border:1px solid #ddd;">

<font color=497890 size=3>

اهداف: پس از اتمام این آزمایشگاه شما قادر خواهید بود

یک سیستم توصیه‌گر با استفاده از پالایش همکاری‌محور $(Collaborative Filtering)$ ایجاد کنید

<font color=GREEN size=5>
HAKAN Fatemi (www.hooko.ir)


____
</div> </div>

سیستم‌های توصیه‌گر مجموعه‌ای از الگوریتم‌ها هستند که برای توصیه اقلام به کاربران، بر اساس اطلاعات گرفته‌شده از کاربر، استفاده می‌شوند

این سیستم‌ها به‌شدت فراگیر شده‌اند و به‌طور معمول در فروشگاه‌های آنلاین، پایگاه‌های داده‌ی فیلم و کاریابی‌ها دیده می‌شوند

در این دفترچه، سیستم‌های توصیه‌گر مبتنی بر محتوا $( \text{Content based recommendation systems} )$ را بررسی کرده

و یک نسخه‌ی ساده از آن را با استفاده از پایتون و کتابخانه‌ی $pandas$ پیاده‌سازی خواهیم کرد

### فهرست مطالب

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ol>
        <li><a href="#ref1">(Acquiring the Data) دریافت داده</a></li>
        <li><a href="#ref2">(Preprocessing) پیش‌پردازش</a></li>
        <li><a href="#ref3">(Content-Based Filtering) پالایش مبتنی بر محتوا</a></li>
    </ol>
</div>
<br>

<a id="ref1"></a>

# دریافت داده

In [1]:
# توجه: در صورت دانلود نشدن، به شکل دستی از آدرس داده شده دانلود کنید
import os
import urllib.request
import zipfile

url= "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%205/data/moviedataset.zip"
if not (os.path.exists("movies.csv") and os.path.exists("ratings.csv")):
    if not os.path.exists("moviedataset.zip"):
        urllib.request.urlretrieve(url, "moviedataset.zip")
    with zipfile.ZipFile("moviedataset.zip", "r") as zip_ref:
        zip_ref.extractall(".")
    print("دانلود و استخراج کامل شد")
else:
    print("فایل‌ها از قبل وجود دارند")

فایل‌ها از قبل وجود دارند


<a id="ref2"></a>

# پیش‌پردازش

ابتدا، بیایید تمام $( \text{import} )$ های مورد نیاز را از سر راه برداریم:

In [2]:
import pandas as pd              # کار با دیتافریم و داده‌های جدولی
from math import sqrt            # محاسبه ریشه دوم
import numpy as np               # عملیات عددی و آرایه‌ای
import matplotlib.pyplot as plt  # رسم نمودار

In [3]:
# توجه داشته باشید: حجم اطلاعات کاربران کمی زیاد و سنگین است

# ذخیره‌ی اطلاعات فیلم‌ها
movies_df = pd.read_csv("movies.csv")
# ذخیره‌ی اطلاعات کاربران
ratings_df = pd.read_csv("ratings.csv")
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


بیایید با استفاده از تابع $( \text{replace} )$ از $( \text{Pandas} )$، سال را از ستون $( \text{title} )$ حذف کرده و آن را در یک ستون جدید به نام $( \text{year} )$ ذخیره کنیم

In [4]:
# برای یافتن سال ذخیره‌شده بین پرانتز (Regular Expressions) استفاده از عبارات منظم
# پرانتزها را مشخص می‌کنیم تا با فیلم‌هایی که در عنوان خود سال دارند تداخل پیدا نکنیم
movies_df["year"] = movies_df.title.str.extract("(\(\d\d\d\d\))", expand=False)
# حذف پرانتزها
movies_df["year"] = movies_df.year.str.extract("(\d\d\d\d)", expand=False)
# حذف سال‌ها از ستون "title"
movies_df["title"] = movies_df.title.str.replace("(\(\d\d\d\d\))", "", regex=True)
# اعمال تابع strip برای حذف هر گونه کاراکتر فضای خالی انتهایی که ممکن است ظاهر شده باشد
movies_df["title"] = movies_df["title"].apply(lambda x: x.strip())
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


با این کار، بیایید مقادیر موجود در ستون $( \text{Genres} )$ را نیز به **لیستی از $( \text{Genres} )$** تقسیم کنیم تا برای استفاده‌های بعدی ساده‌تر شود

این کار را می‌توان با اعمال تابع $( \text{split} )$ رشته‌ای $( \text{Python} )$ بر روی ستون مربوطه انجام داد

In [5]:
# را روی | فراخوانی کنیم split هر ژانر با یک | جدا شده است، بنابراین فقط کافی است تابع 
movies_df["genres"] = movies_df.genres.str.split("|")
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995
2,3,Grumpier Old Men,"[Comedy, Romance]",1995
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995
4,5,Father of the Bride Part II,[Comedy],1995


از آنجا که نگهداری ژانرها در قالب لیست برای تکنیک سیستم توصیه‌گر مبتنی بر محتوا بهینه نیست، از تکنیک $( \text{One Hot Encoding} )$ استفاده خواهیم کرد 

تا لیست ژانرها را به بردارى تبدیل کنیم که در آن هر ستون متناظر با یک مقدار ممکن از ویژگی باشد. این کدگذاری برای تغذیه‌ی داده‌های دسته‌ای $( \text{categorical data} )$ مورد نیاز است

در این حالت، هر ژانر متفاوت را در ستون‌هایی ذخیره می‌کنیم که شامل `1` یا `0` هستند. عدد `1` نشان می‌دهد که یک فیلم آن ژانر را دارد و عدد `0` نشان می‌دهد که ندارد

بیایید این دیتافریم را در متغیر دیگری نیز ذخیره کنیم، زیرا ژانرها برای اولین سیستم توصیه‌گر ما اهمیت چندانی نخواهند داشت

In [6]:
# کپی کردن دیتافریم فیلم‌ها در یک دیتافریم جدید، زیرا در اولین مورد نیازی به استفاده از اطلاعات ژانر نخواهیم داشت
moviesWithGenres_df = movies_df.copy()

# برای هر سطر در دیتافریم، از میان لیست ژانرها عبور کرده و در ستون متناظر مقدار ۱ قرار می‌دهیم
for index, row in movies_df.iterrows():
    for genre in row["genres"]:
        moviesWithGenres_df.at[index, genre] = 1
# با ۰ برای نشان دادن اینکه یک فیلم آن ژانر خاص را ندارد NaN پر کردن مقادیر 
moviesWithGenres_df = moviesWithGenres_df.fillna(0)
moviesWithGenres_df.head()

,movieId,title,genres,year,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Grumpier Old Men,"[Comedy, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Father of the Bride Part II,[Comedy],1995,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


در ادامه، بیایید نگاهی به دیتافریم $( \text{ratings} )$ بیندازیم

In [7]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,169,2.5,1204927694
1,1,2471,3.0,1204927438
2,1,48516,5.0,1204927435
3,2,2571,3.5,1436165433
4,2,109487,4.0,1436165496


هر سطر در دیتافریم $( \text{ratings} )$ دارای یک $( \text{user id} )$ مرتبط با حداقل یک فیلم، یک امتیاز $( \text{rating} )$ و یک $( \text{timestamp} )$ است که نشان می‌دهد کاربر چه زمانی آن را بررسی کرده است

ما به ستون $( \text{timestamp} )$ نیازی نخواهیم داشت، بنابراین بیایید آن را حذف کنیم تا حافظه آزاد شود

In [8]:
# یک سطر یا ستون مشخص را از دیتافریم حذف می‌کند Drop تابع
ratings_df = ratings_df.drop("timestamp", axis=1)
ratings_df.head()

,userId,movieId,rating
0,1,169,2.5
1,1,2471,3.0
2,1,48516,5.0
3,2,2571,3.5
4,2,109487,4.0


<a id="ref3"></a>

# سیستم توصیه‌گر مبتنی بر محتوا

حال، بیایید نگاهی به نحوه‌ی پیاده‌سازی سیستم‌های توصیه‌گر **مبتنی بر محتوا** یا **مبتنی بر $( \text{Item-Item} )$** بیندازیم

این تکنیک سعی می‌کند تشخیص دهد که جنبه‌های مورد علاقه‌ی یک کاربر در یک $( \text{item} )$ چیست و سپس $( \text{item} )$هایی را توصیه می‌کند که آن جنبه‌ها را ارائه می‌دهند

در مورد ما، قصد داریم ژانرهای مورد علاقه‌ی کاربر ورودی را از روی فیلم‌ها و امتیازهای داده‌شده تشخیص دهیم

بیایید با ایجاد یک کاربر ورودی برای توصیه فیلم به او شروع کنیم:

توجه: برای افزودن فیلم‌های بیشتر، کافی است تعداد عناصر موجود در **$( \text{userInput} )$** را افزایش دهید

در افزودن فیلم‌های بیشتر آزادید! فقط مطمئن شوید که آن را با حروف بزرگ بنویسید 

و اگر فیلمی با $( \text{"The"} )$ شروع می‌شود، مانند $( \text{"The Matrix"} )$، آن را به این شکل بنویسید

$( \text{"Matrix, The"} )$

In [9]:
userInput = [
            {"title":"Breakfast Club, The", "rating":5},
            {"title":"Toy Story", "rating":3.5},
            {"title":"Jumanji", "rating":2},
            {"title":"Pulp Fiction", "rating":5},
            {"title":"Akira", "rating":4.5}
         ] 
inputMovies = pd.DataFrame(userInput)
inputMovies

,title,rating
0,"Breakfast Club, The",5.0
1,Toy Story,3.5
2,Jumanji,2.0
3,Pulp Fiction,5.0
4,Akira,4.5


#### افزودن $( \text{movieId} )$ به کاربر ورودی

با تکمیل ورودی، بیایید شناسه‌های $( \text{ID} )$ فیلم‌های ورودی را از دیتافریم $( \text{movies} )$ استخراج کرده و آن‌ها را به آن اضافه کنیم

می‌توانیم این کار را با اول فیلتر کردن سطرهایی که شامل عنوان فیلم ورودی هستند و سپس ادغام این زیرمجموعه با دیتافریم ورودی انجام دهیم

همچنین ستون‌های غیرضروری را برای ورودی حذف می‌کنیم تا حافظه آزاد شود

In [10]:
# فیلتر کردن فیلم‌ها بر اساس عنوان
inputId = movies_df[movies_df["title"].isin(inputMovies["title"].tolist())]
# این کار به‌صورت ضمنی بر اساس عنوان ادغام می‌کند movieId سپس ادغام کردن برای دریافت 
inputMovies = pd.merge(inputId, inputMovies)
# حذف اطلاعاتی که از دیتافریم ورودی استفاده نخواهیم کرد
inputMovies = inputMovies.drop("genres", axis=1).drop("year", axis=1)
# دیتافریم ورودی نهایی
# اگر فیلمی که در بالا اضافه کردید در اینجا نیست، ممکن است در دیتافریم اصلی نباشد یا املای آن متفاوت باشد، لطفاً بزرگ‌نویسی را بررسی کنید
inputMovies

,movieId,title,rating
0,1,Toy Story,3.5
1,2,Jumanji,2.0
2,296,Pulp Fiction,5.0
3,1274,Akira,4.5
4,1968,"Breakfast Club, The",5.0


ما قصد داریم با یادگیری ترجیحات کاربر ورودی شروع کنیم، بنابراین بیایید زیرمجموعه‌ای از فیلم‌هایی را که کاربر ورودی تماشا کرده است

از دیتافریم حاوی ژانرهای تعریف‌شده با مقادیر دودویی $( \text{binary values} )$ دریافت کنیم

In [11]:
# فیلتر کردن فیلم‌ها از ورودی
userMovies = moviesWithGenres_df[moviesWithGenres_df["movieId"].isin(inputMovies["movieId"].tolist())]
userMovies

,movieId,title,genres,year,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
293,296,Pulp Fiction,"[Comedy, Crime, Drama, Thriller]",1994,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1246,1274,Akira,"[Action, Adventure, Animation, Sci-Fi]",1988,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1885,1968,"Breakfast Club, The","[Comedy, Drama]",1985,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


ما فقط به جدول اصلی ژانرها نیاز داریم، پس بیایید این را کمی تمیز کنیم، با بازنشانی ایندکس و حذف ستون‌های 

$( \text{movieId، title، genres و year} )$

In [12]:
# بازنشانی ایندکس برای جلوگیری از مشکلات بعدی
userMovies = userMovies.reset_index(drop=True)
# حذف موارد غیرضروری برای صرفه‌جویی در حافظه و جلوگیری از بروز مشکل
userGenreTable = userMovies.drop("movieId", axis=1).drop("title", axis=1).drop("genres", axis=1).drop("year", axis=1)
userGenreTable

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


اکنون آماده‌ایم تا یادگیری ترجیحات کاربر ورودی را شروع کنیم!

برای انجام این کار، قصد داریم هر ژانر را به وزن‌هایی $( \text{weights} )$ تبدیل کنیم

می‌توانیم این کار را با استفاده از امتیازهای $( \text{reviews} )$ کاربر ورودی و ضرب آن‌ها در جدول ژانرهای کاربر ورودی انجام دهیم و سپس جدول حاصل را بر اساس ستون‌ها جمع‌بندی کنیم

این عملیات در واقع یک ضرب نقطه‌ای $( \text{dot product} )$ بین یک ماتریس و یک بردار است، بنابراین به‌سادگی می‌توانیم با فراخوانی تابع $( \text{"dot"} )$ در پانداس آن را انجام دهیم.

In [13]:
inputMovies["rating"]

0    3.5
1    2.0
2    5.0
3    4.5
4    5.0
Name: rating, dtype: float64

In [14]:
# ضرب نقطه‌ای برای به‌دست آوردن وزن‌ها
userProfile = userGenreTable.transpose().dot(inputMovies["rating"])
# پروفایل کاربر
userProfile

Adventure             10.0
Animation              8.0
Children               5.5
Comedy                13.5
Fantasy                5.5
Romance                0.0
Drama                 10.0
Action                 4.5
Crime                  5.0
Thriller               5.0
Horror                 0.0
Mystery                0.0
Sci-Fi                 4.5
IMAX                   0.0
Documentary            0.0
War                    0.0
Musical                0.0
Western                0.0
Film-Noir              0.0
(no genres listed)     0.0
dtype: float64

اکنون، وزن‌ها را برای هر یک از ترجیحات کاربر داریم. این همان چیزی است که به عنوان $( \text{User Profile} )$ (پروفایل کاربر) شناخته می‌شود

با استفاده از این، می‌توانیم فیلم‌هایی را توصیه کنیم که ترجیحات کاربر را برآورده می‌کنند

بیایید با استخراج جدول ژانرها از دیتافریم اصلی شروع کنیم:

In [15]:
# حال بیایید ژانرهای هر فیلم را در دیتافریم اصلی خود دریافت کنیم
genreTable = moviesWithGenres_df.set_index(moviesWithGenres_df["movieId"])
# و اطلاعات غیرضروری را حذف کنیم
genreTable = genreTable.drop("movieId", axis=1).drop("title", axis=1).drop("genres", axis=1).drop("year", axis=1)
genreTable.head()

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
movieId,,,,,,,,,,,,,,,,,,,,
1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
genreTable.shape

(34208, 20)

با در دست داشتن پروفایل کاربر ورودی و لیست کامل فیلم‌ها و ژانرهای آن‌ها، قصد داریم میانگین وزنی $( \text{weighted average} )$ هر فیلم را 

بر اساس پروفایل ورودی محاسبه کرده و بیست فیلم برتری را که بیشترین تطابق را با آن دارند، توصیه کنیم

In [17]:
# ضرب ژانرها در وزن‌ها و سپس گرفتن میانگین وزنی
recommendationTable_df = ((genreTable*userProfile).sum(axis=1))/(userProfile.sum())
recommendationTable_df.head()

movieId
1    0.594406
2    0.293706
3    0.188811
4    0.328671
5    0.188811
dtype: float64

In [18]:
#Sort our recommendations in descending order
recommendationTable_df = recommendationTable_df.sort_values(ascending=False)
#Just a peek at the values
recommendationTable_df.head()

movieId
5018      0.748252
26093     0.734266
27344     0.720280
148775    0.685315
117646    0.678322
dtype: float64

حالا جدول توصیه‌ها در اینجا آماده است

In [19]:
# جدول نهایی توصیه‌ها
movies_df.loc[movies_df["movieId"].isin(recommendationTable_df.head(20).keys())]

,movieId,title,genres,year
664,673,Space Jam,"[Adventure, Animation, Children, Comedy, Fanta...",1996
1824,1907,Mulan,"[Adventure, Animation, Children, Comedy, Drama...",1998
2902,2987,Who Framed Roger Rabbit?,"[Adventure, Animation, Children, Comedy, Crime...",1988
4923,5018,Motorama,"[Adventure, Comedy, Crime, Drama, Fantasy, Mys...",1991
6793,6902,Interstate 60,"[Adventure, Comedy, Drama, Fantasy, Mystery, S...",2002
8605,26093,"Wonderful World of the Brothers Grimm, The","[Adventure, Animation, Children, Comedy, Drama...",1962
8783,26340,"Twelve Tasks of Asterix, The (Les douze travau...","[Action, Adventure, Animation, Children, Comed...",1976
9296,27344,Revolutionary Girl Utena: Adolescence of Utena...,"[Action, Adventure, Animation, Comedy, Drama, ...",1999
9825,32031,Robots,"[Adventure, Animation, Children, Comedy, Fanta...",2005
11716,51632,Atlantis: Milo's Return,"[Action, Adventure, Animation, Children, Comed...",2003


### مزایا و معایب پالایش مبتنی بر محتوا $( \text{Content-Based Filtering} )$

##### مزایا $( \text{Advantages} )$

*   ترجیحات کاربر را یاد می‌گیرد
*   به‌شدت برای کاربر شخصی‌سازی شده است

##### معایب $( \text{Disadvantages} )$

*   نظرات دیگران را درباره‌ی آیتم در نظر نمی‌گیرد، بنابراین ممکن است توصیه‌های آیتم با کیفیت پایین رخ دهد 

*   استخراج داده‌ها همیشه شهودی نیست 

*   تشخیص این که کاربر چه ویژگی‌هایی از آیتم را دوست دارد یا دوست ندارد، همیشه واضح نیست 


<h2>می‌خواهید بیشتر یاد بگیرید؟</h2>

پلتفرم **هوکو** یک بستر جامع تحلیلی و هوش مصنوعی است که مجموعه‌ای از الگوریتم‌های یادگیری ماشین، ابزارهای تحلیل داده و راهکارهای پیش‌بینی هوشمند را در اختیار شما قرار می‌دهد.
این پلتفرم به شما کمک می‌کند تا تصمیم‌های دقیق‌تر، سریع‌تر و مبتنی بر داده بگیرید؛ 

چه به‌صورت فردی، چه در سطح تیمی و یا در مقیاس سازمانی

___

اکنون می‌توانید نسخه آزمایشی رایگان هوکو را فعال کرده و قدرت هوش مصنوعی را در تصمیم‌گیری‌های خود تجربه کنید

 <a href="https://hooko.ir">HOOKO.ir شروع تجربه در </a>

 ___

## Author

Mahdi Fatemi (HAKAN)

Instagram: @Fatemi_303

09220630140

## web
www.hooko.ir

## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2025-11-06 | 1.0  | HAKAN Fatemi  |  ... |

## <h3 align="center"> © HOOKO.IR Corporation. All rights reserved. <h3/>
